# Comparador de preços de medicamentos — versão 1

MVP para comparar **compra online com retirada**, **compra presencial** e **entrega** nas filiais habituais.

Configuração inicial:
- CEP de referência: `20511-190`
- Produto-teste: **Puran T4 112 µg — 30 comprimidos**
- Drogarias: Pacheco, Raia, Venancio e Drogasmil
- Cadastro com benefício: Raia e Pacheco

> Esta versão registra e compara os preços de forma estruturada. A coleta automática será acrescentada rede por rede depois de validarmos os campos e as regras de comparação.

## 1. Preparar o ambiente
Execute esta célula sempre que o Colab for reiniciado.

In [1]:
from pathlib import Path
from datetime import datetime
from math import ceil

import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

pd.set_option("display.max_colwidth", 120)
print("Ambiente preparado.")

Ambiente preparado.


## 2. Conectar o Google Drive e definir os arquivos
O histórico será preservado no Drive mesmo após o encerramento do Colab.

In [2]:
# Compatível com Google Colab e com ambiente local (Windows/Mac/Linux)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_PROJETO = Path('/content/drive/MyDrive/Comparador_medicamentos')
except ImportError:
    PASTA_PROJETO = Path('.')

PASTA_DADOS = PASTA_PROJETO / 'dados'
PASTA_RELATORIOS = PASTA_PROJETO / 'relatorios'

PASTA_DADOS.mkdir(parents=True, exist_ok=True)
PASTA_RELATORIOS.mkdir(parents=True, exist_ok=True)

ARQUIVO_HISTORICO = PASTA_DADOS / 'historico_precos.csv'
ARQUIVO_DROGARIAS = PASTA_DADOS / 'drogarias.csv'
ARQUIVO_PRODUTOS = PASTA_DADOS / 'produtos.csv'

print(f'Projeto: {PASTA_PROJETO.resolve()}')
print(f'Histórico: {ARQUIVO_HISTORICO.resolve()}')


Mounted at /content/drive
Projeto: /content/drive/MyDrive/Comparador_medicamentos
Histórico: /content/drive/MyDrive/Comparador_medicamentos/dados/historico_precos.csv


## 3. Cadastrar produto e filiais

In [3]:
CEP_REFERENCIA = "20511-190"

produtos = pd.DataFrame([
    {
        "produto_id": "MED001",
        "nome": "Puran T4",
        "principio_ativo": "Levotiroxina sódica",
        "dosagem": "112 µg",
        "apresentacao": "30 comprimidos",
        "fabricante": "Sanofi Aventis",
    }
])

drogarias = pd.DataFrame([
    {
        "drogaria": "Drogarias Pacheco",
        "filial": "Rua Conde de Bonfim, 580 — Tijuca",
        "cadastro_cpf": True,
        "url_produto": "https://www.drogariaspacheco.com.br/puran-t4-112mcg-sanofi-aventis-30-comprimidos/p",
    },
    {
        "drogaria": "Droga Raia",
        "filial": "Rua Conde de Bonfim, 536 — Tijuca",
        "cadastro_cpf": True,
        "url_produto": "https://www.drogaraia.com.br/puran-t4-112mcg-30-comprimidos.html",
    },
    {
        "drogaria": "Drogaria Venancio",
        "filial": "Rua Conde de Bonfim, 532 — Tijuca",
        "cadastro_cpf": False,
        "url_produto": "https://www.drogariavenancio.com.br/",
    },
    {
        "drogaria": "Drogasmil",
        "filial": "Praça Saenz Peña, 29 — Tijuca",
        "cadastro_cpf": False,
        "url_produto": "https://www.drogasmil.com.br/puran-t4-112mcg-30-comprimidos/p",
    },
])

produtos.to_csv(ARQUIVO_PRODUTOS, index=False, encoding="utf-8-sig")
drogarias.to_csv(ARQUIVO_DROGARIAS, index=False, encoding="utf-8-sig")

display(produtos)
display(drogarias[["drogaria", "filial", "cadastro_cpf"]])

,produto_id,nome,principio_ativo,dosagem,apresentacao,fabricante
0,MED001,Puran T4,Levotiroxina sódica,112 µg,30 comprimidos,Sanofi Aventis


,drogaria,filial,cadastro_cpf
0,Drogarias Pacheco,"Rua Conde de Bonfim, 580 — Tijuca",True
1,Droga Raia,"Rua Conde de Bonfim, 536 — Tijuca",True
2,Drogaria Venancio,"Rua Conde de Bonfim, 532 — Tijuca",False
3,Drogasmil,"Praça Saenz Peña, 29 — Tijuca",False


## 4. Carregar ou criar o histórico
Cada registro representa uma condição de preço observada em uma data. Promoções com quantidade mínima permanecem separadas do preço para uma única caixa.

In [4]:
COLUNAS_HISTORICO = [
    "data_hora", "produto_id", "drogaria", "filial",
    "modalidade", "condicao_preco", "valor_total",
    "quantidade_caixas", "custo_por_caixa", "frete",
    "estoque", "fonte", "url", "observacoes"
]

def carregar_historico():
    if ARQUIVO_HISTORICO.exists():
        df = pd.read_csv(ARQUIVO_HISTORICO, encoding="utf-8-sig")
        for coluna in COLUNAS_HISTORICO:
            if coluna not in df.columns:
                df[coluna] = ""
        return df[COLUNAS_HISTORICO].copy()

    df = pd.DataFrame(columns=COLUNAS_HISTORICO)
    df.to_csv(ARQUIVO_HISTORICO, index=False, encoding="utf-8-sig")
    return df

historico = carregar_historico()
print(f"Registros carregados: {len(historico)}")

Registros carregados: 0


## 5. Registrar uma consulta de preço
Informe o **valor total pago ou anunciado** e quantas caixas são exigidas naquela condição. O custo por caixa será calculado automaticamente.

In [11]:
# --------------------------------------------------
# 5. Interface para registrar uma consulta de preço
# --------------------------------------------------

opcoes_drogaria = drogarias["drogaria"].tolist()

mapa_filial = (
    drogarias
    .set_index("drogaria")["filial"]
    .to_dict()
)

mapa_url = (
    drogarias
    .set_index("drogaria")["url_produto"]
    .to_dict()
)


# --------------------------------------------------
# Campos do formulário
# --------------------------------------------------

w_drogaria = widgets.Dropdown(
    options=opcoes_drogaria,
    description="Drogaria:",
    style={"description_width": "120px"}
)

w_modalidade = widgets.Dropdown(
    options=[
        "Online — retirada",
        "Presencial",
        "Online — entrega"
    ],
    description="Modalidade:",
    style={"description_width": "120px"}
)

w_condicao = widgets.Dropdown(
    options=[
        "Preço comum",
        "CPF/clube",
        "Desconto de laboratório",
        "Promoção por quantidade"
    ],
    description="Condição:",
    style={"description_width": "120px"}
)

w_valor_total = widgets.FloatText(
    value=0.0,
    description="Valor total (R$):",
    style={"description_width": "120px"}
)

w_quantidade = widgets.BoundedIntText(
    value=1,
    min=1,
    max=100,
    description="Nº de caixas:",
    style={"description_width": "120px"}
)

w_frete = widgets.FloatText(
    value=0.0,
    description="Frete (R$):",
    style={"description_width": "120px"}
)

w_estoque = widgets.Dropdown(
    options=[
        "Confirmado",
        "Não confirmado",
        "Indisponível"
    ],
    description="Estoque:",
    style={"description_width": "120px"}
)

w_fonte = widgets.Dropdown(
    options=[
        "Site",
        "Aplicativo",
        "Loja física",
        "WhatsApp/telefone",
        "Nota ou comprovante"
    ],
    description="Fonte:",
    style={"description_width": "120px"}
)

w_observacoes = widgets.Textarea(
    placeholder=(
        "Ex.: preço exige CPF; retirada em 30 min; "
        "promoção leve 4 e pague 3"
    ),
    description="Observações:",
    style={"description_width": "120px"},
    layout=widgets.Layout(
        width="700px",
        height="85px"
    )
)


# --------------------------------------------------
# Botão e área de saída
# --------------------------------------------------

botao_salvar = widgets.Button(
    description="Salvar registro",
    button_style="success",
    icon="save"
)

saida_salvar = widgets.Output()


# --------------------------------------------------
# Função para salvar o registro
# --------------------------------------------------

def salvar_registro(_):
    global historico

    with saida_salvar:
        clear_output()

        if w_valor_total.value <= 0:
            print("Informe um valor total maior que zero.")
            return

        custo_por_caixa = (
            w_valor_total.value + w_frete.value
        ) / w_quantidade.value

        novo = {
            "data_hora": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
            "produto_id": "MED001",
            "drogaria": w_drogaria.value,
            "filial": mapa_filial[w_drogaria.value],
            "modalidade": w_modalidade.value,
            "condicao_preco": w_condicao.value,
            "valor_total": round(
                w_valor_total.value,
                2
            ),
            "quantidade_caixas": int(
                w_quantidade.value
            ),
            "custo_por_caixa": round(
                custo_por_caixa,
                2
            ),
            "frete": round(
                w_frete.value,
                2
            ),
            "estoque": w_estoque.value,
            "fonte": w_fonte.value,
            "url": mapa_url[w_drogaria.value],
            "observacoes": w_observacoes.value.strip(),
        }

        if historico.empty:
            historico = pd.DataFrame(
                [novo],
                columns=COLUNAS_HISTORICO
            )
        else:
            historico = pd.concat(
                [
                    historico,
                    pd.DataFrame([novo])
                ],
                ignore_index=True
            )

        historico.to_csv(
            ARQUIVO_HISTORICO,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"Registro salvo: "
            f"{w_drogaria.value} — "
            f"R$ {custo_por_caixa:.2f} por caixa."
        )


# --------------------------------------------------
# Conectar a função ao botão
# --------------------------------------------------

botao_salvar.on_click(salvar_registro)


# --------------------------------------------------
# Exibir o formulário
# --------------------------------------------------

display(
    widgets.VBox(
        [
            w_drogaria,
            w_modalidade,
            w_condicao,

            widgets.HBox(
                [
                    w_valor_total,
                    w_quantidade
                ]
            ),

            widgets.HBox(
                [
                    w_frete,
                    w_estoque
                ]
            ),

            w_fonte,
            w_observacoes,
            botao_salvar,
            saida_salvar
        ]
    )
)

## 6. Comparar as opções
A comparação usa o registro mais recente de cada drogaria, modalidade e condição. Ofertas que exigem mais caixas do que a quantidade desejada são excluídas.

In [22]:
# ============================================================
# CÉLULA 6 — COMPARAÇÃO ATUALIZADA DE PREÇOS
# ============================================================

import pandas as pd
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output


# ------------------------------------------------------------
# 1. Controles
# ------------------------------------------------------------

quantidade_comparacao = widgets.BoundedIntText(
    value=4,
    min=1,
    max=100,
    description="Caixas desejadas:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="330px")
)

somente_estoque_confirmado = widgets.Checkbox(
    value=False,
    description="Mostrar apenas estoque confirmado para retirada",
    indent=False,
    layout=widgets.Layout(width="470px")
)

botao_comparar = widgets.Button(
    description="Comparar",
    button_style="info",
    icon="balance-scale",
    layout=widgets.Layout(
        width="250px",
        height="46px"
    )
)

saida_comparacao = widgets.Output()


# ------------------------------------------------------------
# 2. Funções auxiliares
# ------------------------------------------------------------

def moeda_brasileira(valor):
    """
    Formata um número no padrão monetário brasileiro.
    """

    if pd.isna(valor):
        return "—"

    return (
        f"R$ {float(valor):,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


def estoque_confirmado(valor):
    """
    Considera confirmado apenas o estoque explicitamente
    registrado como disponível para retirada.
    """

    texto = str(valor).strip().lower()

    return texto == "disponível para retirada"


def preparar_historico_comparacao():
    """
    Carrega o histórico e mantém apenas o registro mais recente
    de cada oferta equivalente.
    """

    dados = carregar_historico().copy()

    if dados.empty:
        return dados

    dados["data_hora"] = pd.to_datetime(
        dados["data_hora"],
        errors="coerce"
    )

    colunas_numericas = [
        "valor_total",
        "quantidade_caixas",
        "custo_por_caixa",
        "frete"
    ]

    for coluna in colunas_numericas:
        dados[coluna] = pd.to_numeric(
            dados[coluna],
            errors="coerce"
        )

    dados = dados.dropna(
        subset=[
            "data_hora",
            "custo_por_caixa",
            "quantidade_caixas"
        ]
    )

    dados = dados.sort_values(
        "data_hora",
        ascending=True
    )

    # Uma oferta é identificada pela drogaria, filial,
    # modalidade, condição e quantidade mínima.
    colunas_oferta = [
        "produto_id",
        "drogaria",
        "filial",
        "modalidade",
        "condicao_preco",
        "quantidade_caixas"
    ]

    dados = dados.drop_duplicates(
        subset=colunas_oferta,
        keep="last"
    )

    return dados


# ------------------------------------------------------------
# 3. Comparação
# ------------------------------------------------------------

def executar_comparacao(_):

    with saida_comparacao:

        clear_output(wait=True)

        quantidade_desejada = int(
            quantidade_comparacao.value
        )

        dados = preparar_historico_comparacao()

        if dados.empty:
            print(
                "Ainda não existem preços válidos no histórico."
            )
            return

        ofertas = []

        for _, registro in dados.iterrows():

            condicao = str(
                registro["condicao_preco"]
            ).strip()

            quantidade_registrada = int(
                registro["quantidade_caixas"]
            )

            custo_por_caixa = float(
                registro["custo_por_caixa"]
            )

            frete = (
                float(registro["frete"])
                if pd.notna(registro["frete"])
                else 0.0
            )

            # -----------------------------------------------
            # Preço comum
            # -----------------------------------------------

            if condicao == "Preço comum":

                total_estimado = round(
                    custo_por_caixa
                    * quantidade_desejada
                    + frete,
                    2
                )

                quantidade_minima = 1

            # -----------------------------------------------
            # Promoção por quantidade
            # -----------------------------------------------

            elif condicao == "Promoção por quantidade":

                # A promoção só entra na comparação quando a
                # quantidade desejada é exatamente igual à
                # quantidade registrada.
                #
                # Isso evita supor que a promoção possa ser
                # repetida ou combinada com caixas avulsas.

                if quantidade_desejada != quantidade_registrada:
                    continue

                total_estimado = round(
                    float(registro["valor_total"])
                    + frete,
                    2
                )

                quantidade_minima = quantidade_registrada

            else:

                # Para outras condições ainda não tratadas,
                # usamos a mesma regra conservadora:
                # somente a quantidade exata registrada.

                if quantidade_desejada != quantidade_registrada:
                    continue

                total_estimado = round(
                    float(registro["valor_total"])
                    + frete,
                    2
                )

                quantidade_minima = quantidade_registrada

            disponibilidade_confirmada = estoque_confirmado(
                registro["estoque"]
            )

            if (
                somente_estoque_confirmado.value
                and not disponibilidade_confirmada
            ):
                continue

            ofertas.append(
                {
                    "drogaria": registro["drogaria"],
                    "filial": registro["filial"],
                    "modalidade": registro["modalidade"],
                    "condição": condicao,
                    "R$/caixa": custo_por_caixa,
                    "total estimado": total_estimado,
                    "mínimo de caixas": quantidade_minima,
                    "estoque": registro["estoque"],
                    "confirmação": (
                        "Confirmado"
                        if disponibilidade_confirmada
                        else "Pendente"
                    ),
                    "data_hora": registro["data_hora"],
                    "fonte": registro["fonte"],
                    "observações": registro["observacoes"],
                }
            )

        if not ofertas:

            if somente_estoque_confirmado.value:
                print(
                    "Nenhuma oferta com estoque confirmado "
                    "atende à quantidade escolhida."
                )
                print()
                print(
                    "Desmarque o filtro para visualizar também "
                    "as ofertas com estoque ainda não confirmado."
                )
            else:
                print(
                    "Nenhuma oferta registrada atende à "
                    f"quantidade de {quantidade_desejada} caixas."
                )

            return

        comparacao = pd.DataFrame(ofertas)

        comparacao = comparacao.sort_values(
            by=[
                "total estimado",
                "R$/caixa",
                "drogaria"
            ],
            ascending=[
                True,
                True,
                True
            ]
        ).reset_index(drop=True)

        comparacao.insert(
            0,
            "posição",
            range(1, len(comparacao) + 1)
        )

        # -----------------------------------------------
        # Resumo da melhor opção
        # -----------------------------------------------

        melhor = comparacao.iloc[0]

        display(
            HTML(
                f"""
                <div style="
                    border:2px solid #14b8c8;
                    border-radius:10px;
                    padding:16px;
                    margin:8px 0 18px 0;
                    background:#f3fcfd;
                ">
                    <div style="
                        font-size:18px;
                        font-weight:700;
                        margin-bottom:6px;
                    ">
                        Melhor opção para
                        {quantidade_desejada}
                        {
                            "caixa"
                            if quantidade_desejada == 1
                            else "caixas"
                        }
                    </div>

                    <div style="font-size:17px;">
                        <strong>{melhor["drogaria"]}</strong>
                        — {melhor["condição"]}
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:700;
                        color:#087f8c;
                        margin-top:6px;
                    ">
                        {moeda_brasileira(
                            melhor["total estimado"]
                        )}
                    </div>

                    <div style="
                        margin-top:6px;
                        color:#444;
                    ">
                        {moeda_brasileira(
                            melhor["R$/caixa"]
                        )}
                        por caixa ·
                        Estoque: {melhor["estoque"]}
                    </div>
                </div>
                """
            )
        )

        # -----------------------------------------------
        # Tabela
        # -----------------------------------------------

        tabela = comparacao.copy()

        tabela["data_hora"] = tabela[
            "data_hora"
        ].dt.strftime("%d/%m/%Y %H:%M")

        colunas_exibidas = [
            "posição",
            "drogaria",
            "filial",
            "modalidade",
            "condição",
            "R$/caixa",
            "total estimado",
            "mínimo de caixas",
            "estoque",
            "confirmação",
            "data_hora",
            "fonte",
            "observações"
        ]

        tabela = tabela[colunas_exibidas]

        def destacar_linhas(linha):

            estilos = [""] * len(linha)

            if linha["posição"] == 1:
                estilos = [
                    "background-color: #dff7f9; "
                    "font-weight: 600;"
                ] * len(linha)

            return estilos

        tabela_estilizada = (
            tabela.style
            .hide(axis="index")
            .format(
                {
                    "R$/caixa": moeda_brasileira,
                    "total estimado": moeda_brasileira
                }
            )
            .apply(
                destacar_linhas,
                axis=1
            )
            .set_properties(
                subset=[
                    "posição",
                    "R$/caixa",
                    "total estimado",
                    "mínimo de caixas"
                ],
                **{
                    "text-align": "center"
                }
            )
            .set_properties(
                subset=["observações"],
                **{
                    "white-space": "normal",
                    "min-width": "330px",
                    "max-width": "500px"
                }
            )
            .set_properties(
                subset=["filial"],
                **{
                    "white-space": "normal",
                    "min-width": "220px"
                }
            )
            .set_properties(
                subset=[
                    "drogaria",
                    "modalidade",
                    "condição",
                    "estoque",
                    "confirmação",
                    "fonte"
                ],
                **{
                    "white-space": "normal",
                    "min-width": "130px"
                }
            )
            .set_table_styles(
                [
                    {
                        "selector": "th",
                        "props": [
                            (
                                "background-color",
                                "#f1f5f7"
                            ),
                            (
                                "font-weight",
                                "700"
                            ),
                            (
                                "text-align",
                                "center"
                            ),
                            (
                                "padding",
                                "10px"
                            ),
                            (
                                "border-bottom",
                                "2px solid #c8d3d8"
                            )
                        ]
                    },
                    {
                        "selector": "td",
                        "props": [
                            (
                                "padding",
                                "10px"
                            ),
                            (
                                "border-bottom",
                                "1px solid #dde4e7"
                            ),
                            (
                                "vertical-align",
                                "middle"
                            )
                        ]
                    }
                ]
            )
        )

        display(tabela_estilizada)

        print()
        print(
            "Critério: preços comuns são multiplicados pela "
            "quantidade desejada."
        )
        print(
            "Promoções por quantidade só são utilizadas quando "
            "a quantidade escolhida coincide exatamente com a "
            "regra registrada."
        )
        print(
            "A primeira posição é o menor custo total, mas uma "
            "oferta com estoque pendente ainda precisa ser "
            "confirmada antes da compra."
        )


botao_comparar.on_click(executar_comparacao)


# ------------------------------------------------------------
# 4. Exibição
# ------------------------------------------------------------

display(
    widgets.VBox(
        [
            widgets.HTML(
                """
                <h3>Comparador de preços</h3>
                <p>
                    Escolha a quantidade de caixas e compare as
                    ofertas mais recentes registradas.
                </p>
                """
            ),

            widgets.HBox(
                [
                    quantidade_comparacao,
                    somente_estoque_confirmado
                ],
                layout=widgets.Layout(
                    gap="30px",
                    flex_flow="row wrap"
                )
            ),

            botao_comparar,
            saida_comparacao
        ]
    )
)

## 7. Consultar o histórico e gerar relatório
O relatório é uma cópia ordenada do histórico, pronta para ser aberta no Excel ou no Google Sheets.

In [13]:
botao_historico = widgets.Button(description="Atualizar histórico", icon="refresh")
botao_relatorio = widgets.Button(description="Gerar relatório CSV", button_style="primary", icon="download")
saida_historico = widgets.Output()

def mostrar_historico(_):
    with saida_historico:
        clear_output()
        df = carregar_historico()
        if df.empty:
            print("Histórico vazio.")
        else:
            display(df.sort_values("data_hora", ascending=False).reset_index(drop=True))

def gerar_relatorio(_):
    with saida_historico:
        clear_output()
        df = carregar_historico().sort_values("data_hora", ascending=False)
        nome = f"comparacao_precos_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        destino = PASTA_RELATORIOS / nome
        df.to_csv(destino, index=False, encoding="utf-8-sig")
        print(f"Relatório salvo em: {destino}")

botao_historico.on_click(mostrar_historico)
botao_relatorio.on_click(gerar_relatorio)
display(widgets.VBox([widgets.HBox([botao_historico, botao_relatorio]), saida_historico]))

## 8. Coleta automática — Drogarias Pacheco

In [15]:
# --------------------------------------------------
# 8. Coleta automática — Drogarias Pacheco
# --------------------------------------------------

import json
import re
import requests

from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd


# --------------------------------------------------
# Configuração
# --------------------------------------------------

URL_PACHECO = (
    "https://www.drogariaspacheco.com.br/"
    "puran-t4-112mcg-sanofi-aventis-30-comprimidos/p"
)

PRODUTO_ESPERADO = "Puran T4"
DOSAGEM_ESPERADA = "112mcg"
APRESENTACAO_ESPERADA = "30 Comprimidos"

CABECALHOS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "pt-BR,pt;q=0.9"
}


# --------------------------------------------------
# Função de coleta
# --------------------------------------------------

def consultar_preco_pacheco():
    """
    Consulta a página pública do produto e extrai
    os dados estruturados presentes em skuJson_0.

    A função não faz login, não informa CPF e não
    adiciona o produto ao carrinho.
    """

    try:
        resposta = requests.get(
            URL_PACHECO,
            headers=CABECALHOS,
            timeout=30
        )

        resposta.raise_for_status()

    except requests.exceptions.Timeout:
        raise RuntimeError(
            "A consulta à Pacheco ultrapassou "
            "o tempo limite de 30 segundos."
        )

    except requests.exceptions.HTTPError as erro:
        codigo = erro.response.status_code

        raise RuntimeError(
            f"A Pacheco retornou o código HTTP {codigo}."
        )

    except requests.exceptions.RequestException as erro:
        raise RuntimeError(
            f"Não foi possível acessar a Pacheco: {erro}"
        )

    html = resposta.text

    # Procurar o bloco estruturado do produto
    padrao = (
        r"var\s+skuJson_0\s*=\s*"
        r"(\{.*?\})"
        r";\s*CATALOG_SDK"
    )

    correspondencia = re.search(
        padrao,
        html,
        flags=re.DOTALL
    )

    if correspondencia is None:
        raise RuntimeError(
            "A página foi acessada, mas o bloco "
            "estruturado skuJson_0 não foi encontrado. "
            "A estrutura do site pode ter mudado."
        )

    try:
        dados_produto = json.loads(
            correspondencia.group(1)
        )

    except json.JSONDecodeError as erro:
        raise RuntimeError(
            "O bloco do produto foi encontrado, "
            "mas não pôde ser interpretado."
        ) from erro

    nome_produto = str(
        dados_produto.get("name", "")
    )

    # Conferir se o site devolveu exatamente
    # o produto esperado
    verificacoes = [
        PRODUTO_ESPERADO.lower()
        in nome_produto.lower(),

        DOSAGEM_ESPERADA.lower()
        in nome_produto.lower(),

        APRESENTACAO_ESPERADA.lower()
        in nome_produto.lower()
    ]

    if not all(verificacoes):
        raise RuntimeError(
            "O produto retornado pela página não "
            "corresponde ao Puran T4 112 mcg "
            "com 30 comprimidos."
        )

    skus = dados_produto.get("skus", [])

    if not skus:
        raise RuntimeError(
            "O produto foi identificado, mas nenhum "
            "SKU foi encontrado."
        )

    sku = skus[0]

    preco_online = sku.get("bestPrice")
    preco_referencia = sku.get("listPrice")

    if preco_online is None:
        raise RuntimeError(
            "O preço online não foi encontrado "
            "no bloco estruturado."
        )

    # Os preços são armazenados pelo site em centavos
    preco_online = preco_online / 100

    if preco_referencia is not None:
        preco_referencia = preco_referencia / 100

    data_hora = datetime.now(
        ZoneInfo("America/Sao_Paulo")
    ).strftime("%Y-%m-%d %H:%M:%S")

    resultado = {
        "data_hora": data_hora,
        "drogaria": "Drogarias Pacheco",
        "produto": nome_produto,
        "sku": str(sku.get("sku", "")),
        "preco_online": round(preco_online, 2),
        "preco_referencia": (
            round(preco_referencia, 2)
            if preco_referencia is not None
            else None
        ),
        "disponivel_site": bool(
            sku.get(
                "available",
                dados_produto.get("available", False)
            )
        ),
        "quantidade_informada_site": (
            sku.get("availablequantity")
        ),
        "url": URL_PACHECO
    }

    return resultado


# --------------------------------------------------
# Executar o teste
# --------------------------------------------------

try:
    resultado_pacheco = consultar_preco_pacheco()

    tabela_pacheco = pd.DataFrame(
        [resultado_pacheco]
    )

    display(
        tabela_pacheco.style
        .hide(axis="index")
        .format(
            {
                "preco_online": "R$ {:.2f}",
                "preco_referencia": "R$ {:.2f}"
            }
        )
    )

    print()
    print("Consulta automática concluída.")
    print(
        "Este resultado ainda não foi salvo "
        "no histórico."
    )
    print(
        "O estoque exibido é geral do site, "
        "não a confirmação da filial da Tijuca."
    )

except Exception as erro:
    print("A consulta automática não foi concluída.")
    print()
    print(f"Motivo: {erro}")


data_hora,drogaria,produto,sku,preco_online,preco_referencia,disponivel_site,quantidade_informada_site,url
2026-09-21 10:46:32,Drogarias Pacheco,Puran T4 Levotiroxina Sódica 112mcg 30 Comprimidos,116319,R$ 29.35,R$ 36.58,True,99999,https://www.drogariaspacheco.com.br/puran-t4-112mcg-sanofi-aventis-30-comprimidos/p



Consulta automática concluída.
Este resultado ainda não foi salvo no histórico.
O estoque exibido é geral do site, não a confirmação da filial da Tijuca.


## 9. Salvar automaticamente o preço da Pacheco

In [16]:
# --------------------------------------------------
# 9. Salvar automaticamente o preço da Pacheco
# --------------------------------------------------

botao_coletar_pacheco = widgets.Button(
    description="Consultar e salvar Pacheco",
    button_style="primary",
    icon="cloud-download",
    layout=widgets.Layout(width="260px")
)

saida_coleta_pacheco = widgets.Output()


# --------------------------------------------------
# Verificar duplicação
# --------------------------------------------------

def registro_equivalente_ja_existe(
    df,
    novo_registro
):
    """
    Considera duplicado um registro da mesma data,
    drogaria, modalidade, condição e preço.
    """

    if df.empty:
        return False

    dados = df.copy()

    dados["valor_total_numerico"] = pd.to_numeric(
        dados["valor_total"],
        errors="coerce"
    )

    mesma_data = (
        dados["data_hora"]
        .astype(str)
        .str[:10]
        == novo_registro["data_hora"][:10]
    )

    mesma_drogaria = (
        dados["drogaria"]
        == novo_registro["drogaria"]
    )

    mesma_modalidade = (
        dados["modalidade"]
        == novo_registro["modalidade"]
    )

    mesma_condicao = (
        dados["condicao_preco"]
        == novo_registro["condicao_preco"]
    )

    mesmo_preco = (
        dados["valor_total_numerico"]
        .round(2)
        == round(
            novo_registro["valor_total"],
            2
        )
    )

    duplicados = dados[
        mesma_data
        & mesma_drogaria
        & mesma_modalidade
        & mesma_condicao
        & mesmo_preco
    ]

    return not duplicados.empty


# --------------------------------------------------
# Consultar e salvar
# --------------------------------------------------

def coletar_e_salvar_pacheco(_):
    global historico

    with saida_coleta_pacheco:
        clear_output()

        print(
            "Consultando o preço público "
            "da Drogarias Pacheco..."
        )

        try:
            consulta = consultar_preco_pacheco()

        except Exception as erro:
            print()
            print(
                "A consulta automática não "
                "foi concluída."
            )
            print(f"Motivo: {erro}")
            return

        preco_online = consulta["preco_online"]
        preco_referencia = consulta[
            "preco_referencia"
        ]

        if preco_referencia is not None:
            texto_referencia = (
                f"Preço de referência: "
                f"R$ {preco_referencia:.2f}. "
            )
        else:
            texto_referencia = (
                "Preço de referência não informado. "
            )

        novo = {
            "data_hora": consulta["data_hora"],
            "produto_id": "MED001",
            "drogaria": "Drogarias Pacheco",
            "filial": mapa_filial[
                "Drogarias Pacheco"
            ],
            "modalidade": "Online — retirada",
            "condicao_preco": "Preço comum",
            "valor_total": round(
                preco_online,
                2
            ),
            "quantidade_caixas": 1,
            "custo_por_caixa": round(
                preco_online,
                2
            ),
            "frete": 0.0,

            # A página informa disponibilidade geral,
            # mas não confirma estoque na filial escolhida.
            "estoque": "Não confirmado",

            "fonte": "Site",
            "url": consulta["url"],
            "observacoes": (
                "Coleta automática do preço público. "
                f"{texto_referencia}"
                f"SKU: {consulta['sku']}. "
                "Disponibilidade geral do site: "
                f"{consulta['disponivel_site']}. "
                "Retirada na filial da Tijuca "
                "ainda não confirmada."
            )
        }

        historico = carregar_historico()

        if registro_equivalente_ja_existe(
            historico,
            novo
        ):
            print()
            print(
                "O preço foi consultado com sucesso, "
                "mas um registro equivalente já existe "
                "no histórico de hoje."
            )
            print()
            print(
                f"Preço encontrado: "
                f"R$ {preco_online:.2f}"
            )
            print(
                "Nenhum registro duplicado foi criado."
            )
            return

        if historico.empty:
            historico = pd.DataFrame(
                [novo],
                columns=COLUNAS_HISTORICO
            )
        else:
            historico = pd.concat(
                [
                    historico,
                    pd.DataFrame([novo])
                ],
                ignore_index=True
            )

        historico.to_csv(
            ARQUIVO_HISTORICO,
            index=False,
            encoding="utf-8-sig"
        )

        print()
        print(
            "Consulta automática concluída "
            "e salva no histórico."
        )
        print(
            f"Preço encontrado: "
            f"R$ {preco_online:.2f}"
        )
        print(
            "Estoque da filial: não confirmado."
        )


# --------------------------------------------------
# Conectar função ao botão
# --------------------------------------------------

botao_coletar_pacheco.on_click(
    coletar_e_salvar_pacheco
)


# --------------------------------------------------
# Exibir interface
# --------------------------------------------------

display(
    widgets.VBox(
        [
            widgets.HTML(
                value=(
                    "<b>Drogarias Pacheco</b><br>"
                    "Consulta o preço público do produto "
                    "e registra apenas valores ainda não "
                    "salvos na mesma data."
                )
            ),
            botao_coletar_pacheco,
            saida_coleta_pacheco
        ]
    )
)

## 10. Coleta automática — Drogasmil

In [23]:
# --------------------------------------------------
# 10. Coleta automática — Drogasmil
# --------------------------------------------------

import json
import requests

from bs4 import BeautifulSoup
from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd
import ipywidgets as widgets

from IPython.display import display, clear_output


# --------------------------------------------------
# Configuração
# --------------------------------------------------

URL_BUSCA_DROGASMIL = (
    "https://www.drogasmil.com.br/"
    "puran%20t4?_q=Puran%20T4&map=ft"
)

URL_PRODUTO_DROGASMIL = (
    "https://www.drogasmil.com.br/"
    "puran-t4-112mcg-30-comprimidos/p"
)

NOME_ESPERADO_DROGASMIL = (
    "Puran T4 112mcg 30 Comprimidos"
)

CABECALHOS_DROGASMIL = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "pt-BR,pt;q=0.9"
}


# --------------------------------------------------
# Localizar o produto nos dados estruturados
# --------------------------------------------------

def localizar_produto_drogasmil(dados):
    """
    Procura o produto exato dentro de um bloco
    JSON-LD da Drogasmil.
    """

    if isinstance(dados, list):
        for item in dados:
            encontrado = localizar_produto_drogasmil(
                item
            )

            if encontrado is not None:
                return encontrado

        return None

    if not isinstance(dados, dict):
        return None

    tipo = dados.get("@type")

    # Bloco do próprio produto
    if tipo == "Product":
        nome = str(
            dados.get("name", "")
        ).strip()

        if (
            nome.lower()
            == NOME_ESPERADO_DROGASMIL.lower()
        ):
            return dados

    # Lista de resultados
    elementos = dados.get(
        "itemListElement",
        []
    )

    for elemento in elementos:
        if not isinstance(elemento, dict):
            continue

        produto = elemento.get(
            "item",
            elemento
        )

        encontrado = localizar_produto_drogasmil(
            produto
        )

        if encontrado is not None:
            return encontrado

    return None


# --------------------------------------------------
# Consultar preço
# --------------------------------------------------

def consultar_preco_drogasmil():
    """
    Consulta a página pública de resultados da
    Drogasmil e extrai os dados JSON-LD do produto.

    Não faz login, não informa CPF e não adiciona
    o produto ao carrinho.
    """

    try:
        resposta = requests.get(
            URL_BUSCA_DROGASMIL,
            headers=CABECALHOS_DROGASMIL,
            timeout=30
        )

        resposta.raise_for_status()

    except requests.exceptions.Timeout:
        raise RuntimeError(
            "A consulta à Drogasmil ultrapassou "
            "o tempo limite de 30 segundos."
        )

    except requests.exceptions.HTTPError as erro:
        codigo = erro.response.status_code

        raise RuntimeError(
            f"A Drogasmil retornou o código "
            f"HTTP {codigo}."
        )

    except requests.exceptions.RequestException as erro:
        raise RuntimeError(
            "Não foi possível acessar a Drogasmil: "
            f"{erro}"
        )

    soup = BeautifulSoup(
        resposta.text,
        "html.parser"
    )

    scripts_json = soup.find_all(
        "script",
        attrs={"type": "application/ld+json"}
    )

    if not scripts_json:
        raise RuntimeError(
            "A página foi acessada, mas nenhum "
            "bloco JSON-LD foi encontrado."
        )

    produto_encontrado = None

    for script in scripts_json:
        conteudo = script.string

        if not conteudo:
            continue

        try:
            dados = json.loads(conteudo)

        except json.JSONDecodeError:
            continue

        produto_encontrado = (
            localizar_produto_drogasmil(
                dados
            )
        )

        if produto_encontrado is not None:
            break

    if produto_encontrado is None:
        raise RuntimeError(
            "O Puran T4 112 mcg com 30 comprimidos "
            "não foi encontrado nos dados "
            "estruturados da Drogasmil."
        )

    nome_produto = str(
        produto_encontrado.get(
            "name",
            ""
        )
    ).strip()

    ofertas = produto_encontrado.get(
        "offers",
        {}
    )

    preco = ofertas.get("lowPrice")

    if preco is None:
        lista_ofertas = ofertas.get(
            "offers",
            []
        )

        if lista_ofertas:
            preco = lista_ofertas[0].get(
                "price"
            )

    if preco is None:
        raise RuntimeError(
            "O produto foi identificado, mas o "
            "preço público não foi encontrado."
        )

    try:
        preco = float(preco)

    except (TypeError, ValueError):
        raise RuntimeError(
            "O preço foi encontrado, mas não pôde "
            "ser convertido em número."
        )

    lista_ofertas = ofertas.get(
        "offers",
        []
    )

    if lista_ofertas:
        primeira_oferta = lista_ofertas[0]

        disponibilidade_texto = str(
            primeira_oferta.get(
                "availability",
                ""
            )
        )

    else:
        disponibilidade_texto = ""

    disponivel_site = (
        "InStock"
        in disponibilidade_texto
    )

    data_hora = datetime.now(
        ZoneInfo("America/Sao_Paulo")
    ).strftime("%Y-%m-%d %H:%M:%S")

    resultado = {
        "data_hora": data_hora,
        "drogaria": "Drogasmil",
        "produto": nome_produto,
        "sku": str(
            produto_encontrado.get(
                "sku",
                ""
            )
        ),
        "referencia": str(
            produto_encontrado.get(
                "mpn",
                ""
            )
        ),
        "preco_online": round(
            preco,
            2
        ),
        "disponivel_site": disponivel_site,
        "url": URL_PRODUTO_DROGASMIL
    }

    return resultado


# --------------------------------------------------
# Botão e área de saída
# --------------------------------------------------

botao_coletar_drogasmil = widgets.Button(
    description="Consultar e salvar Drogasmil",
    button_style="primary",
    icon="cloud-download",
    layout=widgets.Layout(width="280px")
)

saida_coleta_drogasmil = widgets.Output()


# --------------------------------------------------
# Consultar e salvar no histórico
# --------------------------------------------------

def coletar_e_salvar_drogasmil(_):
    global historico

    with saida_coleta_drogasmil:
        clear_output()

        print(
            "Consultando o preço público "
            "da Drogasmil..."
        )

        try:
            consulta = consultar_preco_drogasmil()

        except Exception as erro:
            print()
            print(
                "A consulta automática não "
                "foi concluída."
            )
            print(f"Motivo: {erro}")
            return

        preco_online = consulta[
            "preco_online"
        ]

        novo = {
            "data_hora": consulta["data_hora"],
            "produto_id": "MED001",
            "drogaria": "Drogasmil",
            "filial": mapa_filial[
                "Drogasmil"
            ],
            "modalidade": "Online — retirada",
            "condicao_preco": "Preço comum",
            "valor_total": round(
                preco_online,
                2
            ),
            "quantidade_caixas": 1,
            "custo_por_caixa": round(
                preco_online,
                2
            ),
            "frete": 0.0,

            # O site informa disponibilidade geral,
            # não o estoque da filial selecionada.
            "estoque": "Não confirmado",

            "fonte": "Site",
            "url": consulta["url"],
            "observacoes": (
                "Coleta automática do preço público. "
                f"SKU: {consulta['sku']}. "
                f"Referência: {consulta['referencia']}. "
                "Disponibilidade geral do site: "
                f"{consulta['disponivel_site']}. "
                "O site exibe a mensagem "
                "“leve 4 e pague 3”, mas essa promoção "
                "não foi registrada automaticamente. "
                "Retirada na filial da Praça Saenz Peña "
                "ainda não confirmada."
            )
        }

        historico = carregar_historico()

        if registro_equivalente_ja_existe(
            historico,
            novo
        ):
            print()
            print(
                "O preço foi consultado com sucesso, "
                "mas um registro equivalente já existe "
                "no histórico de hoje."
            )
            print()
            print(
                f"Preço encontrado: "
                f"R$ {preco_online:.2f}"
            )
            print(
                "Nenhum registro duplicado foi criado."
            )
            return

        if historico.empty:
            historico = pd.DataFrame(
                [novo],
                columns=COLUNAS_HISTORICO
            )

        else:
            historico = pd.concat(
                [
                    historico,
                    pd.DataFrame([novo])
                ],
                ignore_index=True
            )

        historico.to_csv(
            ARQUIVO_HISTORICO,
            index=False,
            encoding="utf-8-sig"
        )

        print()
        print(
            "Consulta automática concluída "
            "e salva no histórico."
        )
        print(
            f"Preço encontrado: "
            f"R$ {preco_online:.2f}"
        )
        print(
            "Estoque da filial: não confirmado."
        )


# --------------------------------------------------
# Conectar função ao botão
# --------------------------------------------------

botao_coletar_drogasmil.on_click(
    coletar_e_salvar_drogasmil
)


# --------------------------------------------------
# Exibir interface
# --------------------------------------------------

display(
    widgets.VBox(
        [
            widgets.HTML(
                value=(
                    "<b>Drogasmil</b><br>"
                    "Consulta o preço público do produto "
                    "e registra apenas valores ainda não "
                    "salvos na mesma data."
                )
            ),
            botao_coletar_drogasmil,
            saida_coleta_drogasmil
        ]
    )
)

In [24]:
# ============================================================
# CÉLULA 11 — DROGA RAIA: CONSULTA ASSISTIDA E REGISTRO
# ============================================================

import pandas as pd
import ipywidgets as widgets

from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import display, HTML, clear_output


# ------------------------------------------------------------
# 1. Configurações
# ------------------------------------------------------------

URL_RAIA = (
    "https://www.drogaraia.com.br/"
    "puran-t4-112mcg-30-comprimidos.html"
)

DROGARIA_RAIA = "Droga Raia"

FILIAL_RAIA = mapa_filial.get(
    DROGARIA_RAIA,
    "Rua Conde de Bonfim, 536 — Tijuca"
)


# ------------------------------------------------------------
# 2. Funções auxiliares
# ------------------------------------------------------------

def registro_raia_ja_existe(
    historico_atual,
    condicao_preco,
    custo_por_caixa,
    quantidade_caixas
):
    """
    Verifica se um registro equivalente da Raia já foi
    salvo na data atual.
    """

    if historico_atual.empty:
        return False

    dados = historico_atual.copy()

    datas = pd.to_datetime(
        dados["data_hora"],
        errors="coerce"
    ).dt.date

    custos = pd.to_numeric(
        dados["custo_por_caixa"],
        errors="coerce"
    )

    quantidades = pd.to_numeric(
        dados["quantidade_caixas"],
        errors="coerce"
    )

    data_hoje = datetime.now(
        ZoneInfo("America/Sao_Paulo")
    ).date()

    equivalentes = dados[
        (datas == data_hoje)
        & (
            dados["drogaria"]
            .astype(str)
            .str.strip()
            == DROGARIA_RAIA
        )
        & (
            dados["filial"]
            .astype(str)
            .str.strip()
            == FILIAL_RAIA
        )
        & (
            dados["modalidade"]
            .astype(str)
            .str.strip()
            == "Online — retirada"
        )
        & (
            dados["condicao_preco"]
            .astype(str)
            .str.strip()
            == condicao_preco
        )
        & (
            (custos - float(custo_por_caixa))
            .abs()
            < 0.001
        )
        & (
            quantidades
            == int(quantidade_caixas)
        )
    ]

    return not equivalentes.empty


def acrescentar_registro_raia(
    historico_atual,
    novo_registro
):
    """
    Acrescenta um registro mantendo a ordem correta
    das colunas do histórico.
    """

    nova_linha = pd.DataFrame(
        [novo_registro],
        columns=COLUNAS_HISTORICO
    )

    if historico_atual.empty:
        return nova_linha.copy()

    return pd.concat(
        [historico_atual, nova_linha],
        ignore_index=True
    )


def formatar_reais(valor):
    """
    Formata número como moeda brasileira.
    """

    return (
        f"R$ {valor:,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


# ------------------------------------------------------------
# 3. Campos da interface
# ------------------------------------------------------------

preco_comum_raia = widgets.FloatText(
    value=29.59,
    description="Preço comum:",
    step=0.01,
    style={"description_width": "150px"},
    layout=widgets.Layout(width="350px")
)

registrar_promocao_raia = widgets.Checkbox(
    value=True,
    description="Registrar promoção por quantidade"
)

quantidade_promocao_raia = widgets.BoundedIntText(
    value=4,
    min=2,
    max=100,
    description="Caixas da promoção:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="350px")
)

preco_promocional_raia = widgets.FloatText(
    value=26.09,
    description="Preço por caixa:",
    step=0.01,
    style={"description_width": "150px"},
    layout=widgets.Layout(width="350px")
)

estoque_raia = widgets.Dropdown(
    options=[
        "Não confirmado",
        "Disponível para retirada",
        "Indisponível para retirada"
    ],
    value="Não confirmado",
    description="Estoque na filial:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="420px")
)

valores_conferidos_raia = widgets.Checkbox(
    value=False,
    description=(
        "Conferi estes valores no site após informar o CEP"
    ),
    indent=False,
    layout=widgets.Layout(width="550px")
)

botao_salvar_raia = widgets.Button(
    description="Salvar preços conferidos da Raia",
    button_style="info",
    icon="save",
    layout=widgets.Layout(
        width="380px",
        height="48px"
    )
)

saida_raia = widgets.Output()


# ------------------------------------------------------------
# 4. Controle visual da promoção
# ------------------------------------------------------------

def atualizar_campos_promocao_raia(change=None):

    habilitar = registrar_promocao_raia.value

    quantidade_promocao_raia.disabled = not habilitar
    preco_promocional_raia.disabled = not habilitar


registrar_promocao_raia.observe(
    atualizar_campos_promocao_raia,
    names="value"
)

atualizar_campos_promocao_raia()


# ------------------------------------------------------------
# 5. Salvamento
# ------------------------------------------------------------

def salvar_precos_raia(_):

    with saida_raia:

        clear_output(wait=True)

        if not valores_conferidos_raia.value:
            print(
                "Antes de salvar, abra o site da Raia, informe "
                "o CEP e confira os preços."
            )
            print(
                "Depois marque a caixa de confirmação."
            )
            return

        preco_comum = round(
            float(preco_comum_raia.value),
            2
        )

        if preco_comum <= 0:
            print(
                "O preço comum precisa ser maior que zero."
            )
            return

        if not 5 <= preco_comum <= 200:
            print(
                "O preço comum está fora da faixa esperada."
            )
            print(
                "Confira se o valor foi digitado corretamente."
            )
            return

        historico_atual = carregar_historico()

        agora = datetime.now(
            ZoneInfo("America/Sao_Paulo")
        ).strftime("%Y-%m-%d %H:%M:%S")

        registros_salvos = []
        registros_duplicados = []

        # ----------------------------------------------------
        # Preço comum
        # ----------------------------------------------------

        duplicado_comum = registro_raia_ja_existe(
            historico_atual=historico_atual,
            condicao_preco="Preço comum",
            custo_por_caixa=preco_comum,
            quantidade_caixas=1
        )

        if duplicado_comum:

            registros_duplicados.append(
                "Preço comum: "
                f"{formatar_reais(preco_comum)}"
            )

        else:

            novo_comum = {
                "data_hora": agora,
                "produto_id": "MED001",
                "drogaria": DROGARIA_RAIA,
                "filial": FILIAL_RAIA,
                "modalidade": "Online — retirada",
                "condicao_preco": "Preço comum",
                "valor_total": preco_comum,
                "quantidade_caixas": 1,
                "custo_por_caixa": preco_comum,
                "frete": 0.0,
                "estoque": estoque_raia.value,
                "fonte": "Site — conferência manual",
                "url": URL_RAIA,
                "observacoes": (
                    "Preço conferido manualmente no site da "
                    "Droga Raia após informar o CEP. "
                    "Benefícios vinculados ao CPF podem alterar "
                    "o valor apresentado no carrinho."
                ),
            }

            historico_atual = acrescentar_registro_raia(
                historico_atual,
                novo_comum
            )

            registros_salvos.append(
                "Preço comum: "
                f"{formatar_reais(preco_comum)} por caixa"
            )

        # ----------------------------------------------------
        # Promoção por quantidade
        # ----------------------------------------------------

        if registrar_promocao_raia.value:

            quantidade = int(
                quantidade_promocao_raia.value
            )

            preco_promocional = round(
                float(preco_promocional_raia.value),
                2
            )

            if preco_promocional <= 0:
                print(
                    "O preço promocional precisa ser maior "
                    "que zero."
                )
                return

            if preco_promocional >= preco_comum:
                print(
                    "O preço promocional não é menor que o "
                    "preço comum."
                )
                print(
                    "Confira os valores antes de salvar."
                )
                return

            total_promocao = round(
                preco_promocional * quantidade,
                2
            )

            duplicado_promocao = registro_raia_ja_existe(
                historico_atual=historico_atual,
                condicao_preco="Promoção por quantidade",
                custo_por_caixa=preco_promocional,
                quantidade_caixas=quantidade
            )

            if duplicado_promocao:

                registros_duplicados.append(
                    "Promoção: "
                    f"{quantidade} caixas por "
                    f"{formatar_reais(preco_promocional)} cada"
                )

            else:

                novo_promocional = {
                    "data_hora": agora,
                    "produto_id": "MED001",
                    "drogaria": DROGARIA_RAIA,
                    "filial": FILIAL_RAIA,
                    "modalidade": "Online — retirada",
                    "condicao_preco": (
                        "Promoção por quantidade"
                    ),
                    "valor_total": total_promocao,
                    "quantidade_caixas": quantidade,
                    "custo_por_caixa": preco_promocional,
                    "frete": 0.0,
                    "estoque": estoque_raia.value,
                    "fonte": "Site — conferência manual",
                    "url": URL_RAIA,
                    "observacoes": (
                        "Promoção conferida manualmente no "
                        "site da Droga Raia após informar o CEP. "
                        f"Oferta para {quantidade} caixas, com "
                        f"custo de "
                        f"{formatar_reais(preco_promocional)} "
                        "por caixa. A aplicação definitiva deve "
                        "ser confirmada no carrinho. Benefícios "
                        "vinculados ao CPF podem alterar o valor."
                    ),
                }

                historico_atual = acrescentar_registro_raia(
                    historico_atual,
                    novo_promocional
                )

                registros_salvos.append(
                    "Promoção: "
                    f"{quantidade} caixas por "
                    f"{formatar_reais(preco_promocional)} cada "
                    f"— total {formatar_reais(total_promocao)}"
                )

        # ----------------------------------------------------
        # Gravação do histórico
        # ----------------------------------------------------

        if registros_salvos:

            historico_atual = historico_atual[
                COLUNAS_HISTORICO
            ]

            historico_atual.to_csv(
                ARQUIVO_HISTORICO,
                index=False,
                encoding="utf-8-sig"
            )

        # ----------------------------------------------------
        # Resultado
        # ----------------------------------------------------

        if registros_salvos:

            print("Registros salvos com sucesso:")

            for registro in registros_salvos:
                print(f"• {registro}")

        if registros_duplicados:

            print()

            print(
                "Registros equivalentes que já existiam hoje:"
            )

            for registro in registros_duplicados:
                print(f"• {registro}")

            print(
                "Nenhuma cópia desses registros foi criada."
            )

        if not registros_salvos and not registros_duplicados:
            print("Nenhum registro foi criado.")


botao_salvar_raia.on_click(salvar_precos_raia)


# ------------------------------------------------------------
# 6. Interface
# ------------------------------------------------------------

display(
    widgets.VBox(
        [
            widgets.HTML(
                f"""
                <h3>Droga Raia — consulta assistida</h3>

                <p>
                    A Raia bloqueia consultas automáticas feitas
                    diretamente pelo Google Colab.
                </p>

                <p>
                    <a
                        href="{URL_RAIA}"
                        target="_blank"
                        style="
                            display:inline-block;
                            padding:12px 20px;
                            background:#1976d2;
                            color:white;
                            text-decoration:none;
                            border-radius:4px;
                            font-weight:600;
                        "
                    >
                        Abrir Puran T4 no site da Raia
                    </a>
                </p>

                <p>
                    No site, informe o CEP <strong>20511-190</strong>,
                    selecione retirada e confira a disponibilidade
                    na filial da Rua Conde de Bonfim, 536.
                </p>

                <p>
                    Os valores abaixo são apenas os últimos
                    observados. Confirme-os antes de salvar.
                </p>
                """
            ),

            preco_comum_raia,
            registrar_promocao_raia,
            quantidade_promocao_raia,
            preco_promocional_raia,
            estoque_raia,

            widgets.HTML(
                "<hr style='margin:14px 0;'>"
            ),

            valores_conferidos_raia,
            botao_salvar_raia,
            saida_raia,
        ]
    )
)

In [25]:
# ============================================================
# CÉLULA 12 — COLETA AUTOMÁTICA: DROGARIA VENANCIO
# ============================================================

import re
import requests
import pandas as pd
import ipywidgets as widgets

from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import display, clear_output


# ------------------------------------------------------------
# 1. Configurações
# ------------------------------------------------------------

API_VENANCIO = (
    "https://www.drogariavenancio.com.br/"
    "api/catalog_system/pub/products/search/puran%20t4"
)

URL_PRODUTO_VENANCIO = (
    "https://www.drogariavenancio.com.br/"
    "puran-t4-112mcg-sanofi-aventis-30-comprimidos/p"
)

DROGARIA_VENANCIO = "Drogaria Venancio"

FILIAL_VENANCIO = mapa_filial.get(
    DROGARIA_VENANCIO,
    "Rua Conde de Bonfim, 532 — Tijuca"
)

EAN_PURAN_112 = "7897595903372"

CABECALHOS_VENANCIO = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json",
    "Accept-Language": "pt-BR,pt;q=0.9"
}


# ------------------------------------------------------------
# 2. Funções auxiliares
# ------------------------------------------------------------

def converter_float_venancio(valor):

    if valor is None:
        return None

    try:
        return float(valor)
    except (TypeError, ValueError):
        return None


def encontrar_produto_venancio(produtos):
    """
    Localiza o produto exato pelo EAN e valida nome,
    dosagem e quantidade de comprimidos.
    """

    for produto in produtos:

        nome = str(
            produto.get("productName", "")
        ).lower()

        produto_compativel = (
            "puran t4" in nome
            and "112mcg" in nome.replace(" ", "")
            and "30 comprimidos" in nome
        )

        for item in produto.get("items", []):

            ean = str(item.get("ean", ""))

            if (
                ean == EAN_PURAN_112
                and produto_compativel
            ):
                return produto, item

    return None, None


def encontrar_oferta_venancio(item):
    """
    Seleciona a oferta do vendedor principal Venancio.
    """

    vendedores = item.get("sellers", [])

    for vendedor in vendedores:

        if vendedor.get("sellerDefault") is True:
            return vendedor.get(
                "commertialOffer",
                {}
            )

    if vendedores:
        return vendedores[0].get(
            "commertialOffer",
            {}
        )

    return {}


def extrair_promocao_venancio(oferta, preco_comum):
    """
    Interpreta somente promoções explicitamente descritas
    como 'LEVE X PAGUE Y'.

    Se a regra não estiver clara, nenhuma promoção é criada.
    """

    promocoes = (
        oferta.get("PromotionTeasers")
        or oferta.get("Teasers")
        or []
    )

    for promocao in promocoes:

        nome_promocao = str(
            promocao.get("Name")
            or promocao.get("<Name>k__BackingField")
            or ""
        ).upper()

        resultado = re.search(
            r"LEVE\s*(\d+)\s*PAGUE\s*(\d+)",
            nome_promocao
        )

        if not resultado:
            continue

        quantidade_leve = int(resultado.group(1))
        quantidade_paga = int(resultado.group(2))

        if (
            quantidade_leve < 2
            or quantidade_paga < 1
            or quantidade_paga >= quantidade_leve
        ):
            continue

        condicoes = (
            promocao.get("Conditions")
            or promocao.get(
                "<Conditions>k__BackingField"
            )
            or {}
        )

        quantidade_minima = (
            condicoes.get("MinimumQuantity")
            or condicoes.get(
                "<MinimumQuantity>k__BackingField"
            )
        )

        if quantidade_minima is not None:

            try:
                quantidade_minima = int(
                    quantidade_minima
                )
            except (TypeError, ValueError):
                quantidade_minima = None

        # A quantidade mínima estruturada precisa ser
        # compatível com o texto da promoção.
        if (
            quantidade_minima is not None
            and quantidade_minima != quantidade_leve
        ):
            continue

        valor_total = round(
            preco_comum * quantidade_paga,
            2
        )

        custo_efetivo = round(
            valor_total / quantidade_leve,
            2
        )

        return {
            "nome": nome_promocao,
            "quantidade_leve": quantidade_leve,
            "quantidade_paga": quantidade_paga,
            "valor_total": valor_total,
            "custo_por_caixa": custo_efetivo
        }

    return None


def consultar_venancio():
    """
    Consulta o catálogo público da Drogaria Venancio.
    """

    resposta = requests.get(
        API_VENANCIO,
        headers=CABECALHOS_VENANCIO,
        timeout=30
    )

    resposta.raise_for_status()

    produtos = resposta.json()

    if not isinstance(produtos, list):
        raise ValueError(
            "O catálogo da Venancio retornou um formato "
            "inesperado."
        )

    produto, item = encontrar_produto_venancio(
        produtos
    )

    if produto is None or item is None:
        raise ValueError(
            "O Puran T4 112 mcg com 30 comprimidos "
            "não foi encontrado no catálogo."
        )

    oferta = encontrar_oferta_venancio(item)

    preco_comum = converter_float_venancio(
        oferta.get("Price")
    )

    preco_referencia = converter_float_venancio(
        oferta.get("ListPrice")
    )

    quantidade_geral = oferta.get(
        "AvailableQuantity"
    )

    disponivel_geral = bool(
        oferta.get("IsAvailable")
    )

    if preco_comum is None or preco_comum <= 0:
        raise ValueError(
            "O preço atual não pôde ser identificado."
        )

    if not 5 <= preco_comum <= 200:
        raise ValueError(
            f"O valor encontrado, R$ {preco_comum:.2f}, "
            "está fora da faixa esperada."
        )

    promocao = extrair_promocao_venancio(
        oferta=oferta,
        preco_comum=preco_comum
    )

    return {
        "nome": produto.get("productName"),
        "produto_id_site": produto.get("productId"),
        "sku": item.get("itemId"),
        "ean": item.get("ean"),
        "url": (
            produto.get("link")
            or URL_PRODUTO_VENANCIO
        ),
        "preco_comum": round(preco_comum, 2),
        "preco_referencia": (
            round(preco_referencia, 2)
            if preco_referencia is not None
            else None
        ),
        "disponivel_geral": disponivel_geral,
        "quantidade_geral": quantidade_geral,
        "promocao": promocao
    }


def registro_venancio_ja_existe(
    historico_atual,
    condicao_preco,
    custo_por_caixa,
    quantidade_caixas
):
    """
    Verifica se um registro equivalente já foi salvo
    na data atual.
    """

    if historico_atual.empty:
        return False

    dados = historico_atual.copy()

    datas = pd.to_datetime(
        dados["data_hora"],
        errors="coerce"
    ).dt.date

    custos = pd.to_numeric(
        dados["custo_por_caixa"],
        errors="coerce"
    )

    quantidades = pd.to_numeric(
        dados["quantidade_caixas"],
        errors="coerce"
    )

    data_hoje = datetime.now(
        ZoneInfo("America/Sao_Paulo")
    ).date()

    equivalentes = dados[
        (datas == data_hoje)
        & (
            dados["drogaria"]
            .astype(str)
            .str.strip()
            == DROGARIA_VENANCIO
        )
        & (
            dados["filial"]
            .astype(str)
            .str.strip()
            == FILIAL_VENANCIO
        )
        & (
            dados["modalidade"]
            .astype(str)
            .str.strip()
            == "Online — retirada"
        )
        & (
            dados["condicao_preco"]
            .astype(str)
            .str.strip()
            == condicao_preco
        )
        & (
            (custos - float(custo_por_caixa))
            .abs()
            < 0.001
        )
        & (
            quantidades == int(quantidade_caixas)
        )
    ]

    return not equivalentes.empty


def acrescentar_registro_venancio(
    historico_atual,
    novo_registro
):
    """
    Acrescenta um registro mantendo a ordem das colunas.
    """

    nova_linha = pd.DataFrame(
        [novo_registro],
        columns=COLUNAS_HISTORICO
    )

    if historico_atual.empty:
        return nova_linha.copy()

    return pd.concat(
        [historico_atual, nova_linha],
        ignore_index=True
    )


def formatar_reais_venancio(valor):

    return (
        f"R$ {float(valor):,.2f}"
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )


# ------------------------------------------------------------
# 3. Consulta e salvamento
# ------------------------------------------------------------

def consultar_e_salvar_venancio(_):

    with saida_venancio:

        clear_output(wait=True)

        print(
            "Consultando o catálogo público da "
            "Drogaria Venancio..."
        )

        try:

            resultado = consultar_venancio()

            historico_atual = carregar_historico()

            agora = datetime.now(
                ZoneInfo("America/Sao_Paulo")
            ).strftime("%Y-%m-%d %H:%M:%S")

            registros_salvos = []
            registros_duplicados = []

            preco_comum = resultado["preco_comum"]

            # ------------------------------------------------
            # Preço comum
            # ------------------------------------------------

            duplicado_comum = (
                registro_venancio_ja_existe(
                    historico_atual=historico_atual,
                    condicao_preco="Preço comum",
                    custo_por_caixa=preco_comum,
                    quantidade_caixas=1
                )
            )

            if duplicado_comum:

                registros_duplicados.append(
                    "Preço comum: "
                    f"{formatar_reais_venancio(preco_comum)}"
                )

            else:

                observacao_comum = (
                    "Preço obtido automaticamente no catálogo "
                    "público da Drogaria Venancio. "
                    "A disponibilidade informada pelo catálogo "
                    "é geral e não confirma estoque para retirada "
                    "na filial da Rua Conde de Bonfim."
                )

                if resultado["preco_referencia"] is not None:

                    observacao_comum += (
                        " Preço de referência exibido: "
                        f"{formatar_reais_venancio(
                            resultado['preco_referencia']
                        )}."
                    )

                novo_comum = {
                    "data_hora": agora,
                    "produto_id": "MED001",
                    "drogaria": DROGARIA_VENANCIO,
                    "filial": FILIAL_VENANCIO,
                    "modalidade": "Online — retirada",
                    "condicao_preco": "Preço comum",
                    "valor_total": preco_comum,
                    "quantidade_caixas": 1,
                    "custo_por_caixa": preco_comum,
                    "frete": 0.0,
                    "estoque": "Não confirmado",
                    "fonte": "Site — coleta automática",
                    "url": resultado["url"],
                    "observacoes": observacao_comum
                }

                historico_atual = (
                    acrescentar_registro_venancio(
                        historico_atual,
                        novo_comum
                    )
                )

                registros_salvos.append(
                    "Preço comum: "
                    f"{formatar_reais_venancio(preco_comum)} "
                    "por caixa"
                )

            # ------------------------------------------------
            # Promoção leve X e pague Y
            # ------------------------------------------------

            promocao = resultado["promocao"]

            if promocao is not None:

                quantidade_promocao = promocao[
                    "quantidade_leve"
                ]

                custo_promocional = promocao[
                    "custo_por_caixa"
                ]

                total_promocao = promocao[
                    "valor_total"
                ]

                duplicado_promocao = (
                    registro_venancio_ja_existe(
                        historico_atual=historico_atual,
                        condicao_preco=(
                            "Promoção por quantidade"
                        ),
                        custo_por_caixa=custo_promocional,
                        quantidade_caixas=quantidade_promocao
                    )
                )

                if duplicado_promocao:

                    registros_duplicados.append(
                        "Promoção: "
                        f"{quantidade_promocao} caixas por "
                        f"{formatar_reais_venancio(
                            custo_promocional
                        )} cada"
                    )

                else:

                    novo_promocional = {
                        "data_hora": agora,
                        "produto_id": "MED001",
                        "drogaria": DROGARIA_VENANCIO,
                        "filial": FILIAL_VENANCIO,
                        "modalidade": "Online — retirada",
                        "condicao_preco": (
                            "Promoção por quantidade"
                        ),
                        "valor_total": total_promocao,
                        "quantidade_caixas": (
                            quantidade_promocao
                        ),
                        "custo_por_caixa": (
                            custo_promocional
                        ),
                        "frete": 0.0,
                        "estoque": "Não confirmado",
                        "fonte": "Site — coleta automática",
                        "url": resultado["url"],
                        "observacoes": (
                            f"Promoção {promocao['nome']} "
                            "identificada automaticamente no "
                            "catálogo público. "
                            f"Leve {quantidade_promocao} e pague "
                            f"{promocao['quantidade_paga']}. "
                            f"Total calculado: "
                            f"{formatar_reais_venancio(
                                total_promocao
                            )}. "
                            "A aplicação definitiva da promoção "
                            "e a retirada na filial precisam ser "
                            "confirmadas no carrinho."
                        )
                    }

                    historico_atual = (
                        acrescentar_registro_venancio(
                            historico_atual,
                            novo_promocional
                        )
                    )

                    registros_salvos.append(
                        "Promoção: "
                        f"{quantidade_promocao} caixas por "
                        f"{formatar_reais_venancio(
                            custo_promocional
                        )} cada — total "
                        f"{formatar_reais_venancio(
                            total_promocao
                        )}"
                    )

            # ------------------------------------------------
            # Salvar arquivo
            # ------------------------------------------------

            if registros_salvos:

                historico_atual = historico_atual[
                    COLUNAS_HISTORICO
                ]

                historico_atual.to_csv(
                    ARQUIVO_HISTORICO,
                    index=False,
                    encoding="utf-8-sig"
                )

            # ------------------------------------------------
            # Resultado
            # ------------------------------------------------

            print()
            print(f"Produto: {resultado['nome']}")
            print(f"EAN: {resultado['ean']}")

            print(
                "Preço comum encontrado: "
                f"{formatar_reais_venancio(preco_comum)}"
            )

            if resultado["preco_referencia"] is not None:

                print(
                    "Preço de referência: "
                    f"{formatar_reais_venancio(
                        resultado['preco_referencia']
                    )}"
                )

            if promocao is not None:

                print(
                    "Promoção encontrada: "
                    f"leve {promocao['quantidade_leve']} e "
                    f"pague {promocao['quantidade_paga']}"
                )

                print(
                    "Total promocional calculado: "
                    f"{formatar_reais_venancio(
                        promocao['valor_total']
                    )}"
                )

            print()

            if registros_salvos:

                print("Registros salvos:")

                for item in registros_salvos:
                    print(f"• {item}")

            if registros_duplicados:

                print()
                print(
                    "Registros equivalentes que já existiam "
                    "hoje:"
                )

                for item in registros_duplicados:
                    print(f"• {item}")

                print(
                    "Nenhuma cópia desses registros foi criada."
                )

            print()
            print(
                "A disponibilidade geral do catálogo não "
                "confirma retirada na filial da Tijuca."
            )

        except requests.RequestException as erro:

            print()
            print(
                "Não foi possível acessar o catálogo da "
                "Drogaria Venancio."
            )
            print(f"Detalhes técnicos: {erro}")

        except Exception as erro:

            print()
            print(
                "O catálogo foi acessado, mas os dados não "
                "puderam ser interpretados com segurança."
            )
            print(f"Detalhes técnicos: {erro}")


# ------------------------------------------------------------
# 4. Interface
# ------------------------------------------------------------

botao_venancio = widgets.Button(
    description="Consultar e salvar Venancio",
    button_style="info",
    icon="search",
    layout=widgets.Layout(
        width="360px",
        height="48px"
    )
)

saida_venancio = widgets.Output()

botao_venancio.on_click(
    consultar_e_salvar_venancio
)

display(
    widgets.VBox(
        [
            widgets.HTML(
                """
                <h3>Drogaria Venancio</h3>

                <p>
                    Consulta automaticamente o catálogo público
                    e registra o preço comum e uma promoção
                    “leve X e pague Y”, quando a regra estiver
                    explicitamente informada.
                </p>
                """
            ),
            botao_venancio,
            saida_venancio
        ]
    )
)

# preço presencial

In [26]:
# ============================================================
# CÉLULA 13 — REGISTRO DE PREÇO NA LOJA FÍSICA
# ============================================================

import pandas as pd
import ipywidgets as widgets

from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import display, clear_output


# ------------------------------------------------------------
# 1. Farmácias participantes
# ------------------------------------------------------------

FILIAIS_PRESENCIAIS = {
    "Drogarias Pacheco": (
        "Rua Conde de Bonfim, 580 — Tijuca"
    ),
    "Droga Raia": (
        "Rua Conde de Bonfim, 536 — Tijuca"
    ),
    "Drogaria Venancio": (
        "Rua Conde de Bonfim, 532 — Tijuca"
    ),
    "Drogasmil": (
        "Praça Saenz Peña, 29 — Tijuca"
    )
}


# ------------------------------------------------------------
# 2. Campos
# ------------------------------------------------------------

drogaria_presencial = widgets.Dropdown(
    options=list(FILIAIS_PRESENCIAIS.keys()),
    value="Drogaria Venancio",
    description="Drogaria:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="480px")
)

filial_presencial = widgets.Text(
    value=FILIAIS_PRESENCIAIS[
        "Drogaria Venancio"
    ],
    description="Filial:",
    disabled=True,
    style={"description_width": "150px"},
    layout=widgets.Layout(width="680px")
)

preco_presencial = widgets.FloatText(
    value=0.0,
    description="Preço por caixa:",
    step=0.01,
    style={"description_width": "150px"},
    layout=widgets.Layout(width="350px")
)

quantidade_presencial = widgets.BoundedIntText(
    value=1,
    min=1,
    max=100,
    description="Quantidade:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="350px")
)

usou_cpf_presencial = widgets.Checkbox(
    value=False,
    description="Preço obtido com CPF cadastrado",
    indent=False,
    layout=widgets.Layout(width="400px")
)

estoque_presencial = widgets.Dropdown(
    options=[
        "Disponível na loja",
        "Indisponível na loja",
        "Não confirmado"
    ],
    value="Disponível na loja",
    description="Estoque:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="420px")
)

forma_consulta_presencial = widgets.Dropdown(
    options=[
        "Consulta presencial",
        "Consulta por telefone",
        "Consulta por WhatsApp"
    ],
    value="Consulta presencial",
    description="Como conferiu:",
    style={"description_width": "150px"},
    layout=widgets.Layout(width="450px")
)

observacoes_presencial = widgets.Textarea(
    value="",
    placeholder=(
        "Ex.: preço informado no caixa; desconto aplicado "
        "após informar o CPF; promoção válida somente hoje."
    ),
    description="Observações:",
    style={"description_width": "150px"},
    layout=widgets.Layout(
        width="700px",
        height="100px"
    )
)

confirmacao_presencial = widgets.Checkbox(
    value=False,
    description=(
        "Confirmo que este preço foi verificado para "
        "Puran T4 112 mcg, caixa com 30 comprimidos"
    ),
    indent=False,
    layout=widgets.Layout(width="720px")
)

botao_salvar_presencial = widgets.Button(
    description="Salvar preço da loja física",
    button_style="info",
    icon="save",
    layout=widgets.Layout(
        width="350px",
        height="48px"
    )
)

saida_presencial = widgets.Output()


# ------------------------------------------------------------
# 3. Atualização automática do endereço
# ------------------------------------------------------------

def atualizar_filial_presencial(change=None):

    farmacia = drogaria_presencial.value

    filial_presencial.value = (
        FILIAIS_PRESENCIAIS[farmacia]
    )


drogaria_presencial.observe(
    atualizar_filial_presencial,
    names="value"
)


# ------------------------------------------------------------
# 4. Verificação de duplicidade
# ------------------------------------------------------------

def registro_presencial_ja_existe(
    historico_atual,
    drogaria,
    filial,
    custo_por_caixa,
    quantidade_caixas
):
    """
    Verifica se o mesmo preço presencial já foi registrado
    hoje para a mesma drogaria e filial.
    """

    if historico_atual.empty:
        return False

    dados = historico_atual.copy()

    datas = pd.to_datetime(
        dados["data_hora"],
        errors="coerce"
    ).dt.date

    custos = pd.to_numeric(
        dados["custo_por_caixa"],
        errors="coerce"
    )

    quantidades = pd.to_numeric(
        dados["quantidade_caixas"],
        errors="coerce"
    )

    data_hoje = datetime.now(
        ZoneInfo("America/Sao_Paulo")
    ).date()

    equivalentes = dados[
        (datas == data_hoje)
        & (
            dados["drogaria"]
            .astype(str)
            .str.strip()
            == drogaria
        )
        & (
            dados["filial"]
            .astype(str)
            .str.strip()
            == filial
        )
        & (
            dados["modalidade"]
            .astype(str)
            .str.strip()
            == "Loja física"
        )
        & (
            dados["condicao_preco"]
            .astype(str)
            .str.strip()
            == "Preço comum"
        )
        & (
            (custos - float(custo_por_caixa))
            .abs()
            < 0.001
        )
        & (
            quantidades == int(quantidade_caixas)
        )
    ]

    return not equivalentes.empty


# ------------------------------------------------------------
# 5. Salvamento
# ------------------------------------------------------------

def salvar_preco_presencial(_):

    with saida_presencial:

        clear_output(wait=True)

        if not confirmacao_presencial.value:

            print(
                "Marque a confirmação de que o preço "
                "corresponde ao medicamento correto."
            )
            return

        preco = round(
            float(preco_presencial.value),
            2
        )

        quantidade = int(
            quantidade_presencial.value
        )

        if preco <= 0:

            print(
                "Informe um preço por caixa maior que zero."
            )
            return

        if not 5 <= preco <= 200:

            print(
                "O preço informado está fora da faixa "
                "esperada."
            )
            print(
                "Confira se o valor digitado corresponde "
                "ao preço de uma caixa."
            )
            return

        drogaria = drogaria_presencial.value
        filial = filial_presencial.value

        historico_atual = carregar_historico()

        duplicado = registro_presencial_ja_existe(
            historico_atual=historico_atual,
            drogaria=drogaria,
            filial=filial,
            custo_por_caixa=preco,
            quantidade_caixas=quantidade
        )

        if duplicado:

            print(
                "Um registro presencial equivalente já "
                "existe no histórico de hoje."
            )
            print(
                "Nenhum registro duplicado foi criado."
            )
            return

        agora = datetime.now(
            ZoneInfo("America/Sao_Paulo")
        ).strftime("%Y-%m-%d %H:%M:%S")

        valor_total = round(
            preco * quantidade,
            2
        )

        observacoes = []

        if usou_cpf_presencial.value:

            observacoes.append(
                "Preço obtido com CPF cadastrado."
            )
        else:

            observacoes.append(
                "Preço obtido sem benefício de CPF informado."
            )

        observacoes.append(
            f"Forma de conferência: "
            f"{forma_consulta_presencial.value}."
        )

        texto_adicional = (
            observacoes_presencial.value.strip()
        )

        if texto_adicional:
            observacoes.append(texto_adicional)

        novo_registro = {
            "data_hora": agora,
            "produto_id": "MED001",
            "drogaria": drogaria,
            "filial": filial,
            "modalidade": "Loja física",
            "condicao_preco": "Preço comum",
            "valor_total": valor_total,
            "quantidade_caixas": quantidade,
            "custo_por_caixa": preco,
            "frete": 0.0,
            "estoque": estoque_presencial.value,
            "fonte": forma_consulta_presencial.value,
            "url": "",
            "observacoes": " ".join(observacoes)
        }

        nova_linha = pd.DataFrame(
            [novo_registro],
            columns=COLUNAS_HISTORICO
        )

        if historico_atual.empty:

            historico_atual = nova_linha.copy()

        else:

            historico_atual = pd.concat(
                [
                    historico_atual,
                    nova_linha
                ],
                ignore_index=True
            )

        historico_atual = historico_atual[
            COLUNAS_HISTORICO
        ]

        historico_atual.to_csv(
            ARQUIVO_HISTORICO,
            index=False,
            encoding="utf-8-sig"
        )

        preco_formatado = (
            f"R$ {preco:,.2f}"
            .replace(",", "X")
            .replace(".", ",")
            .replace("X", ".")
        )

        total_formatado = (
            f"R$ {valor_total:,.2f}"
            .replace(",", "X")
            .replace(".", ",")
            .replace("X", ".")
        )

        print("Registro presencial salvo com sucesso.")
        print()
        print(f"Drogaria: {drogaria}")
        print(f"Filial: {filial}")
        print(f"Preço por caixa: {preco_formatado}")
        print(f"Quantidade conferida: {quantidade}")
        print(f"Valor total: {total_formatado}")
        print(f"Estoque: {estoque_presencial.value}")

        if usou_cpf_presencial.value:
            print("Benefício de CPF: sim")
        else:
            print("Benefício de CPF: não informado")

        # Limpa somente os campos que devem ser preenchidos
        # novamente em uma próxima consulta.
        preco_presencial.value = 0.0
        observacoes_presencial.value = ""
        confirmacao_presencial.value = False


botao_salvar_presencial.on_click(
    salvar_preco_presencial
)


# ------------------------------------------------------------
# 6. Interface
# ------------------------------------------------------------

display(
    widgets.VBox(
        [
            widgets.HTML(
                """
                <h3>Preço na loja física</h3>

                <p>
                    Use este formulário depois de consultar o
                    preço diretamente na filial, por telefone
                    ou por WhatsApp.
                </p>

                <p>
                    Informe o valor de uma caixa. O comparador
                    calculará automaticamente o total para a
                    quantidade desejada.
                </p>
                """
            ),

            drogaria_presencial,
            filial_presencial,
            preco_presencial,
            quantidade_presencial,
            usou_cpf_presencial,
            estoque_presencial,
            forma_consulta_presencial,
            observacoes_presencial,
            confirmacao_presencial,
            botao_salvar_presencial,
            saida_presencial
        ]
    )
)